In [1]:
!pip install gradio

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

from google.colab import files

**Upload** Dataset

In [ ]:
uploaded = files.upload()

df = pd.read_csv("fake_job_postings.csv")

df = df[["description", "fraudulent"]]
df["description"] = df["description"].fillna("")

**NLP Feature Engineering**

In [ ]:
X_text = df["description"].astype(str).values
y = df["fraudulent"].values

tfidf = TfidfVectorizer(max_features=6000, stop_words="english")
X = tfidf.fit_transform(X_text).toarray()

**Train-Test Split**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)

class_weights = dict(enumerate(class_weights))

In [ ]:
model = Sequential([
    Dense(512, activation="relu", input_shape=(6000,)),
    Dropout(0.5),

    Dense(256, activation="relu"),
    Dropout(0.4),

    Dense(128, activation="relu"),
    Dropout(0.3),

    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

**Train Model**

In [ ]:
model.fit(
    X_train, y_train,
    epochs=12,
    batch_size=32,
    validation_split=0.2,
    class_weight=class_weights
)

Evaluation

In [ ]:
loss, acc = model.evaluate(X_test, y_test)
print("🔥 Accuracy:", acc)

y_pred = (model.predict(X_test) > 0.5).astype(int)

print("\n📊 Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\n📄 Classification Report:")
print(classification_report(y_test, y_pred))

Smart Prediction Function

In [ ]:
def predict_job(text):
    vec = tfidf.transform([text]).toarray()
    pred = model.predict(vec)[0][0]

    print("\n🧠 AI Score:", pred)

    if pred > 0.7:
        print("❌ Very Likely FAKE JOB")
    elif pred > 0.4:
        print("⚠️ Suspicious Job")
    else:
        print("✅ Likely REAL JOB")

**Test Example**

In [ ]:
predict_job("Work from home job. No experience needed. Earn 5000 dollars easily.")

**Web App**

In [ ]:
import gradio as gr

def predict_ui(text):
    vec = tfidf.transform([text]).toarray()
    pred = model.predict(vec)[0][0]

    if pred > 0.7:
        return "❌ FAKE JOB"
    elif pred > 0.4:
        return "⚠️ SUSPICIOUS JOB"
    else:
        return "✅ REAL JOB"

app = gr.Interface(
    fn=predict_ui,
    inputs="text",
    outputs="text",
    title="Fake Job Detection AI",
    description="Paste job description and check if it's real or fake"
)

app.launch()